# F1 · Linaje de datos — **100% dinámico** (lee los parquet reales)

🎯 **Qué hace:** genera el linaje de F1 con `src/linaje`. **No contiene ningún número ni columna pegada**: filas, columnas y diffs se leen del parquet real. Usa tus constantes `RUTA_INTERIM`/`RUTA_PROCESSED` y **descubre los ficheros con glob**.

📋 **Requisitos:** `src/linaje`, `pyarrow`, `openpyxl`, y los parquet de F1 en disco (`data/01_interim/`, `data/02_processed/`).

📤 **Genera:** `docs/html/linaje/f1_linaje.html` (con conmutador) + descargas `.xlsx/.json/.md` en `data/demo/linaje/` y copia en `docs/html/linaje/descargas/`.

🔄 **Flujo:** descubrir parquet → nivel notebook → nivel función → exportar.

➡️ **Siguiente:** mismo patrón para F2–F7 (solo cambian las constantes de ruta).

In [1]:
import sys, shutil
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import RUTA_INTERIM, RUTA_PROCESSED   # tus constantes de proyecto
from src.linaje import Linaje
from src.linaje.exporters import (a_html_3vistas, a_xlsx_combinado,
                                  a_json_combinado, a_markdown_combinado)
from src.linaje.inventario import parsear_fase, funciones_de

DOCS = ROOT/'docs'/'html'/'linaje'; DESC = DOCS/'descargas'; DATA = ROOT/'data'/'demo'/'linaje'
for d in (DOCS, DESC, DATA): d.mkdir(parents=True, exist_ok=True)

In [2]:
import src.linaje.exporters as ex
print("grafo:", hasattr(ex, '_vista_grafo'))
print("mapa I/O:", hasattr(ex, '_mapa_io'))

grafo: True
mapa I/O: True


## Configuración — SOLO parámetros metodológicos

Las rutas salen de tus constantes y los ficheros se **descubren solos** (`glob`). Lo único declarado: la clave, el nombre de la tabla de preinscripción (entra en el 2.º paso, como en `f1_m04b`) y los *motivos* en texto. **Nada de esto son resultados.**

In [3]:
CLAVE  = 'per_id_ficticio'
PREINS = 'preinscripcion'   # se añade en m04b, no en la unión m04a

# Tablas limpias: TODAS las de data/01_interim/ (descubiertas con glob)
FUENTES = {p.stem: p for p in sorted(RUTA_INTERIM.glob('*.parquet'))}

# Fronteras guardadas por el pipeline F1
INTERMEDIOS = {'df_alumno_base': RUTA_PROCESSED/'df_alumno_base.parquet'}
FINAL       = ('df_alumno',      RUTA_PROCESSED/'df_alumno.parquet')

MOTIVOS = {
    'union':  'm04a: left join de las tablas sobre la espina (expedientes) por clave; '
              'drop_duplicates elimina duplicados exactos.',
    'resto':  'm04b: añade preinscripción (left join). m04c/m04d: corrección de notas y '
              'vía de acceso sobre df_alumno.',
}
print('Tablas limpias descubiertas:', len(FUENTES), '->', list(FUENTES))

# --- diagnóstico: ¿qué hay realmente en disco? ---
print('\nRUTA_INTERIM   =', RUTA_INTERIM, '| existe:', RUTA_INTERIM.exists())
print('RUTA_PROCESSED =', RUTA_PROCESSED, '| existe:', RUTA_PROCESSED.exists())
if RUTA_INTERIM.exists():
    print('  parquet en 01_interim :', [p.name for p in RUTA_INTERIM.glob('*.parquet')])
if RUTA_PROCESSED.exists():
    print('  parquet en 02_processed:', [p.name for p in RUTA_PROCESSED.glob('*.parquet')])
if not FUENTES:
    print('\n⛔ No hay parquet en data/01_interim/. Revisa la ruta o genera las tablas limpias (f1_m02).')

Tablas limpias descubiertas: 9 -> ['becas', 'demograficos', 'domicilios', 'expedientes', 'notas', 'preinscripcion', 'recibos', 'titulaciones', 'trabajo']

RUTA_INTERIM   = C:\FF\AU_UJI_v2\data\01_interim | existe: True
RUTA_PROCESSED = C:\FF\AU_UJI_v2\data\02_processed | existe: True
  parquet en 01_interim : ['becas.parquet', 'demograficos.parquet', 'domicilios.parquet', 'expedientes.parquet', 'notas.parquet', 'preinscripcion.parquet', 'recibos.parquet', 'titulaciones.parquet', 'trabajo.parquet']
  parquet en 02_processed: ['df_alumno.parquet', 'df_alumno_base.parquet']


In [4]:
def cargar(lin, nombre, ruta, clave=CLAVE):
    """Lee el parquet REAL si existe (literal); si no, avisa y registra vacío."""
    if Path(ruta).exists():
        return lin.leer_parquet(nombre, ruta, clave=clave)
    print(f'  ⚠ falta: {nombre} → {ruta}')
    return lin.declarar(nombre, None, None, clave=clave, fichero=str(ruta))

## Nivel *notebook* (cadena entre fronteras reales)

In [5]:
ln = Linaje('fase1','notebook','Linaje F1 · Transformación (nivel notebook)')
for n,r in FUENTES.items(): cargar(ln, n, r)
for n,r in INTERMEDIOS.items(): cargar(ln, n, r)
cargar(ln, *FINAL)

# Espina = expedientes (si está); si no, la primera fuente disponible
fuentes_ok = [n for n in FUENTES if ln.datasets[n].n_filas is not None]
espina = 'expedientes' if 'expedientes' in fuentes_ok else (fuentes_ok[0] if fuentes_ok else None)

# Paso 1 (m04a): unión de las tablas limpias EXCEPTO preinscripción → df_alumno_base
entr_union = ([espina] if espina else []) + [n for n in fuentes_ok if n not in (espina, PREINS)]
ln.paso(id='f1_union', notebook='f1_m04a_union_tablas', operacion='merge + dedup',
        funcion='pd.merge / drop_duplicates', entradas=entr_union, salida='df_alumno_base',
        motivo_filas=MOTIVOS['union'])

# Paso 2 (m04b/c/d): df_alumno_base + preinscripción → df_alumno
entr_resto = ['df_alumno_base'] + ([PREINS] if PREINS in fuentes_ok else [])
ln.paso(id='f1_resto', notebook='f1_m04b–d', operacion='merge + corrección',
        funcion='pd.merge / loc', entradas=entr_resto, salida=FINAL[0],
        motivo_filas=MOTIVOS['resto'])

print(f'{len(ln.pasos)} transiciones. Δfilas y columnas: derivados del esquema real.')

2 transiciones. Δfilas y columnas: derivados del esquema real.


## Nivel *función* (procedencia de columnas, modo `aporte`)

Para cada tabla limpia, el motor calcula — leyendo los esquemas reales — qué columnas suyas acaban en `df_alumno` (excluida la clave). Sin restas de filas (no comparables).

In [6]:
lf = Linaje('fase1','funcion','Linaje F1 · Transformación (nivel función)')
for n,r in FUENTES.items(): cargar(lf, n, r)
cargar(lf, *FINAL)

for n in [x for x in FUENTES if lf.datasets[x].n_filas is not None]:
    lf.paso(id=f'aporta_{n}', notebook='f1_m04a_union_tablas', operacion='aporte de columnas',
            funcion="pd.merge(how='left')", entradas=[n], salida=FINAL[0],
            claves_join=[CLAVE], modo='aporte',
            motivo_cols=f"Columnas de '{n}' que persisten en df_alumno.")

print(f'{len(lf.pasos)} aportes. Columnas: derivadas del esquema real de cada parquet.')

9 aportes. Columnas: derivadas del esquema real de cada parquet.


## Exportar (un HTML con conmutador + descargas)

In [7]:
# Inventario de los 14 notebooks de F1 (rol deducido del contenido, dinámico)
NB_FASE1 = ROOT/'notebooks'/'fase1_transformacion'
inventario = parsear_fase(NB_FASE1)
funciones  = funciones_de(inventario)
print(f'{len(inventario)} notebooks · {len(funciones)} funciones reales')

xlsx = a_xlsx_combinado(ln, lf, DATA/'f1_linaje.xlsx', inventario=inventario)
js   = a_json_combinado(ln, lf, DATA/'f1_linaje.json')
md   = a_markdown_combinado(ln, lf, DATA/'f1_linaje.md')
for f in (xlsx, js, md): shutil.copy(f, DESC/f.name)

html = a_html_3vistas(ln, lf, inventario, funciones, DOCS/'f1_linaje.html',
                      ruta_style='../style.css',
                      descargas={'xlsx':'descargas/f1_linaje.xlsx',
                                 'json':'descargas/f1_linaje.json',
                                 'md':'descargas/f1_linaje.md'},
                      titulo='Linaje F1 · Transformación')
print('HTML  :', html.relative_to(ROOT))
print('Excel :', (DATA/'f1_linaje.xlsx').relative_to(ROOT))

14 notebooks · 6 funciones reales
HTML  : docs\html\linaje\f1_linaje.html
Excel : data\demo\linaje\f1_linaje.xlsx
